In [ ]:
from radio_diffusion.paths import get_path

image_dir = get_path("MOJAVE_FITS_DIR")
uv_dir = get_path("MOJAVE_UV_DIR")

image_glob = "*.icn.fits.gz"
uv_suffix = ".uvf"

include_sources = []  # Empty means all sources.
exclude_sources = []
source_pattern = "*"  # Filename-style source filter, for example "0003*".

shuffle = True
random_seed = 42
n = 12  # Number of selected images to process; -1 means all selected images.
workers = 12

extent_beam_modes = ["native", "circular"]  # Either mode or both for jet extent.

# Inner-jet direction from CLEAN components.
orientation_settings = {
    "inner_mas": 0.3,
    "outer_mas": 1.0,
    "sigma_cut": 3.0,
}

# Image-based ridgeline detection and radial extent.
ridgeline_settings = {
    "radial_step_mas": 0.04,
    "max_radius_mas": None,  # None uses the image boundary.
    "half_width_deg": 90.0,
    "angle_step_deg": 0.5,
    "sigma_cut": 8.0,
}
initial_pa_override_deg = None  # None uses measured PA; a number seeds tracing for every selected image.
ridge_interpolation_step_mas = 0.05  # Display sampling; does not change the detected endpoint.

csv_path = get_path("OUTPUT_ROOT") / "mojave_conditioning_metrics.csv"


In [ ]:
import pandas as pd

if not image_dir.is_dir():
    raise FileNotFoundError(f"Image directory not found: {image_dir}")
if not uv_dir.is_dir():
    raise FileNotFoundError(f"Visibility directory not found: {uv_dir}")

image_files = sorted(path for path in image_dir.glob(image_glob) if path.is_file())
image_records = []
for image_path in image_files:
    observation_key = image_path.name.removesuffix(".icn.fits.gz")
    source_id, band, date_text = observation_key.split(".")
    uv_path = uv_dir / f"{observation_key}{uv_suffix}"
    image_records.append({
        "image_filename": image_path.name,
        "source": source_id,
        "observation_date": date_text.replace("_", "-"),
        "visibility_filename": uv_path.name if uv_path.is_file() else None,
    })

file_index = pd.DataFrame(image_records, columns=[
    "image_filename", "source", "observation_date", "visibility_filename",
])
matched_count = int(file_index["visibility_filename"].notna().sum())
print(f"Image directory: {image_dir}")
print(f"Visibility directory: {uv_dir}")
print(f"Images found: {len(file_index)}")
print(f"Images with matching visibility data: {matched_count}")
print(f"Images without matching visibility data: {len(file_index) - matched_count}")


In [ ]:
import numpy as np
from astropy.io import fits


def load_image(filename):
    """Load one image and its primary header from the configured image directory."""
    path = image_dir / filename
    with fits.open(path, memmap=False) as hdul:
        image = np.squeeze(hdul[0].data).copy()
        header = hdul[0].header.copy()

    if image.ndim != 2:
        raise ValueError(f"Expected a 2D image in {filename}; got shape {image.shape}")

    return image, header


In [ ]:
from fnmatch import fnmatchcase

if not isinstance(n, int) or (n != -1 and n < 1):
    raise ValueError("n must be a positive integer or -1.")

selected_files = file_index.copy()
if include_sources:
    selected_files = selected_files[selected_files["source"].isin(include_sources)]
selected_files = selected_files[~selected_files["source"].isin(exclude_sources)]
selected_files = selected_files[
    selected_files["source"].map(lambda source_id: fnmatchcase(source_id, source_pattern))
]
eligible_count = len(selected_files)

if shuffle:
    selected_files = selected_files.sample(frac=1, random_state=random_seed)
if n != -1:
    selected_files = selected_files.head(n)
selected_files = selected_files.reset_index(drop=True)

print(f"Images after source filtering: {eligible_count}")
print(f"Selected images: {len(selected_files)}")
display(selected_files)


In [ ]:
# Total VLBA flux: sum of all CLEAN-model component fluxes, including negatives.
# This follows Kovalev et al. (2005), Section 4, without a threshold or aperture.
# https://arxiv.org/pdf/astro-ph/0505536
# AIPS Memo 117, Section 6.1: FLUX contains component total flux in Jy.
# https://www.aips.nrao.edu/TEXT/PUBL/AIPSMEM117.PDF

def total_flux_jy(filename):
    """Return the signed sum of HDU 1 AIPS CC component fluxes in Jy."""
    with fits.open(image_dir / filename, memmap=False) as hdul:
        if len(hdul) < 2 or hdul[1].header.get("EXTNAME", "").strip().upper() != "AIPS CC":
            raise ValueError(f"Expected an AIPS CC table in HDU 1: {filename}")
        cc_hdu = hdul[1]
        if "FLUX" not in cc_hdu.columns.names:
            raise ValueError(f"Missing FLUX column in HDU 1: {filename}")
        unit = (cc_hdu.columns["FLUX"].unit or "").strip()
        if unit.upper() != "JY":
            raise ValueError(f"Expected FLUX units of Jy in {filename}; got {unit!r}")
        if cc_hdu.data is None or len(cc_hdu.data) == 0:
            raise ValueError(f"Empty AIPS CC table: {filename}")

        fluxes = np.asarray(cc_hdu.data["FLUX"], dtype=np.float64)
        if not np.isfinite(fluxes).all():
            raise ValueError(f"Non-finite CLEAN component flux in {filename}")
        return float(fluxes.sum(dtype=np.float64))


In [ ]:
# Peak brightness is the maximum pixel value in the supplied CLEAN image.
# Jy/beam is the image brightness convention used in Kovalev et al. (2005), Fig. 1.
# https://arxiv.org/pdf/astro-ph/0505536
# NaN pixels are treated as blanks; no interpolation or Gaussian fit is applied.

def peak_brightness_jy_per_beam(image, header):
    """Return the maximum non-blank HDU 0 pixel value in Jy/beam."""
    unit = str(header.get("BUNIT", "")).strip()
    if unit.replace(" ", "").upper() != "JY/BEAM":
        raise ValueError(f"Expected image units of Jy/beam; got {unit!r}")
    if np.isinf(image).any() or not np.isfinite(image).any():
        raise ValueError("Image contains infinite values or has no finite pixels.")

    return float(np.nanmax(image))


In [ ]:
from astropy.wcs import WCS

# Lister et al. (2013), Section 5.2: flux-weighted CLEAN-component PA,
# within 0.15-1 mas of the core and above three times the image rms.
# https://doi.org/10.1088/0004-6256/146/5/120
# Adaptation: the core is fixed at the geometric image centre.
# Numerical choice: a circular mean handles angle wraparound.
# The published cut compares component flux in Jy with image rms in Jy/beam
# numerically; it is not a conversion between these two units.

def jet_orientation_deg(filename, noise_jy_per_beam, inner_mas=0.15, outer_mas=1.0, sigma_cut=3.0):
    """Return inner-jet PA in [0, 360) degrees, north through east; NaN if undefined."""
    if not np.isfinite(noise_jy_per_beam) or noise_jy_per_beam <= 0:
        raise ValueError("Image noise must be finite and positive, in Jy/beam.")
    if not 0 <= inner_mas < outer_mas or sigma_cut <= 0:
        raise ValueError("Require 0 <= inner_mas < outer_mas and sigma_cut > 0.")

    with fits.open(image_dir / filename, memmap=False) as hdul:
        header = hdul[0].header.copy()
        if len(hdul) < 2 or hdul[1].header.get("EXTNAME", "").strip().upper() != "AIPS CC":
            raise ValueError(f"Expected an AIPS CC table in HDU 1: {filename}")
        cc_hdu = hdul[1]
        for column, allowed_units in [
            ("FLUX", {"JY"}),
            ("DELTAX", {"DEG", "DEGREE", "DEGREES"}),
            ("DELTAY", {"DEG", "DEGREE", "DEGREES"}),
        ]:
            if column not in cc_hdu.columns.names:
                raise ValueError(f"Missing {column} column in {filename}")
            if (cc_hdu.columns[column].unit or "").strip().upper() not in allowed_units:
                raise ValueError(f"Unexpected units for {column} in {filename}")
        if cc_hdu.data is None or len(cc_hdu.data) == 0:
            return float("nan")
        flux = np.asarray(cc_hdu.data["FLUX"], dtype=np.float64)
        dx = np.asarray(cc_hdu.data["DELTAX"], dtype=np.float64)
        dy = np.asarray(cc_hdu.data["DELTAY"], dtype=np.float64)

    if not (np.isfinite(flux).all() and np.isfinite(dx).all() and np.isfinite(dy).all()):
        raise ValueError(f"Non-finite CLEAN-component data in {filename}")
    # AIPS CC offsets are in degrees along the image axes, relative to CRPIX.
    x_pixel = dx / header["CDELT1"] + header["CRPIX1"] - 1
    y_pixel = dy / header["CDELT2"] + header["CRPIX2"] - 1
    wcs = WCS(header).celestial
    core_x_pixel = (header["NAXIS1"] - 1) / 2
    core_y_pixel = (header["NAXIS2"] - 1) / 2
    core = wcs.pixel_to_world(core_x_pixel, core_y_pixel)
    positions = wcs.pixel_to_world(x_pixel, y_pixel)
    radius_mas = core.separation(positions).mas
    keep = (
        (radius_mas >= inner_mas) & (radius_mas <= outer_mas)
        & (flux > sigma_cut * noise_jy_per_beam)
    )
    if not keep.any():
        return float("nan")

    angles = core.position_angle(positions[keep]).rad
    weights = flux[keep] / flux[keep].sum()
    east = np.sum(weights * np.sin(angles))
    north = np.sum(weights * np.cos(angles))
    if np.hypot(east, north) <= 1e-12:
        return float("nan")
    return float(np.degrees(np.arctan2(east, north)) % 360)


In [ ]:
# Beam axes are in degrees in FITS; 1 degree = 3,600,000 mas.
# Assumption: header NOISE is in image units, requiring BUNIT = Jy/beam.
# NOISE is labelled theoretical RMS in the inspected sample headers.
# Using it for the Lister et al. image-noise cut is an approximation:
# https://doi.org/10.1088/0004-6256/146/5/120 (Section 5.2).

def beam_noise_metadata(header):
    """Extract restoring-beam parameters and the header theoretical noise estimate."""
    unit = str(header.get("BUNIT", "")).strip()
    if unit.replace(" ", "").upper() != "JY/BEAM":
        raise ValueError(f"Expected image units of Jy/beam; got {unit!r}")
    bmaj_deg = float(header.get("BMAJ", np.nan))
    bmin_deg = float(header.get("BMIN", np.nan))
    bpa_deg = float(header.get("BPA", np.nan))
    try:
        noise = float(header.get("NOISE", np.nan))
    except (TypeError, ValueError):
        noise = float("nan")
    if not np.isfinite(noise) or noise <= 0:
        noise = float("nan")

    return {
        "bmaj_deg": bmaj_deg,
        "bmin_deg": bmin_deg,
        "bpa_deg": bpa_deg,
        "bmaj_mas": bmaj_deg * 3_600_000,
        "bmin_mas": bmin_deg * 3_600_000,
        "image_unit": unit,
        "noise_jy_per_beam": noise,
        "noise_method": "header_theoretical" if np.isfinite(noise) else "unavailable",
        "noise_header_comment": header.comments["NOISE"] if "NOISE" in header else "",
    }


In [ ]:
# Pushkarev et al. (2017), Sections 2.1 and 3.1, use circular-beam images.
# Adaptation: circularise each epoch separately to its own BMAJ.
# https://arxiv.org/pdf/1705.02888
# Gaussian beam variances add under convolution; beam area is proportional
# to BMAJ * BMIN. The new/old beam-area ratio restores Jy/beam scaling.

def circularise_image(image, header):
    """Return a circular-beam image and header; requires finite 2D pixels."""
    metadata = beam_noise_metadata(header)
    bmaj = metadata["bmaj_deg"]
    bmin = metadata["bmin_deg"]
    bpa = metadata["bpa_deg"]
    if not np.isfinite([bmaj, bmin, bpa]).all() or not 0 < bmin <= bmaj:
        raise ValueError("Require finite beam parameters with 0 < BMIN <= BMAJ.")
    if image.ndim != 2 or not np.isfinite(image).all():
        raise ValueError("Circularisation requires a finite 2D image.")

    if bmaj == bmin:
        circular_image = image.copy()
    else:
        # Additional Gaussian smoothing acts along the original minor axis.
        sigma_deg = np.sqrt(bmaj**2 - bmin**2) / np.sqrt(8 * np.log(2))
        pa = np.deg2rad(bpa)
        minor_axis = np.array([np.cos(pa), -np.sin(pa)])  # East, north.
        sky_per_pixel = WCS(header).celestial.pixel_scale_matrix
        sigma_x, sigma_y = np.linalg.solve(sky_per_pixel, sigma_deg * minor_axis)

        # Reflect five projected kernel standard deviations at each edge.
        pad_x = int(np.ceil(5 * abs(sigma_x)))
        pad_y = int(np.ceil(5 * abs(sigma_y)))
        padded = np.pad(image, ((pad_y, pad_y), (pad_x, pad_x)), mode="reflect")
        fx = np.fft.rfftfreq(padded.shape[1])[None, :]
        fy = np.fft.fftfreq(padded.shape[0])[:, None]
        gaussian = np.exp(-2 * np.pi**2 * (sigma_x * fx + sigma_y * fy)**2)
        smoothed = np.fft.irfft2(
            np.fft.rfft2(padded) * gaussian, s=padded.shape, axes=(-2, -1),
        )
        height, width = image.shape
        circular_image = smoothed[pad_y:pad_y + height, pad_x:pad_x + width].copy()
        circular_image *= bmaj / bmin

    circular_header = header.copy()
    circular_header["BMAJ"] = bmaj
    circular_header["BMIN"] = bmaj
    circular_header["BPA"] = 0.0
    circular_header["DATAMAX"] = float(circular_image.max())
    circular_header["DATAMIN"] = float(circular_image.min())
    circular_header.remove("NOISE", ignore_missing=True)
    circular_header.add_history(
        "Circularised to BMAJ with Fourier Gaussian smoothing and Jy/beam rescaling."
    )
    return circular_image, circular_header


In [ ]:
# Pushkarev et al. (2017), Section 2.1: minimum RMS of four corner
# regions, each covering 1/16 of the image area.
# https://arxiv.org/pdf/1705.02888
# Numerical choices: RMS about zero, no clipping; quarter-side lengths
# rounded down. Blank pixels and unusable corner estimates are excluded.

def four_corner_rms_jy_per_beam(image, header):
    """Return minimum positive corner RMS in Jy/beam, or NaN if unavailable."""
    unit = str(header.get("BUNIT", "")).strip()
    if unit.replace(" ", "").upper() != "JY/BEAM":
        raise ValueError(f"Expected image units of Jy/beam; got {unit!r}")
    if image.ndim != 2 or min(image.shape) < 8:
        raise ValueError("Corner RMS estimation requires a 2D image of at least 8 x 8 pixels.")

    height, width = image.shape
    corner_height, corner_width = height // 4, width // 4
    corners = [
        image[:corner_height, :corner_width],
        image[:corner_height, -corner_width:],
        image[-corner_height:, :corner_width],
        image[-corner_height:, -corner_width:],
    ]
    estimates = []
    for corner in corners:
        pixels = corner[np.isfinite(corner)].astype(np.float64)
        if pixels.size < 2:
            continue
        rms = float(np.sqrt(np.mean(pixels**2)))
        if np.isfinite(rms) and rms > 0:
            estimates.append(rms)

    return min(estimates) if estimates else float("nan")


In [ ]:
from astropy import units as u
from scipy.ndimage import map_coordinates

# Pushkarev et al. (2017), Section 3.1: split the above-8-sigma arc
# intensity equally between two sides of the ridge position.
# https://arxiv.org/pdf/1705.02888
# Numerical choices: bilinear interpolation, uniform angular samples,
# and a discrete intensity-weighted median. The angular window defaults
# to +/-90 degrees around the supplied direction; this is configurable.

def sample_jet_arc(
    image, header, radius_mas, noise_jy_per_beam, pa_center_deg,
    half_width_deg=90.0, angle_step_deg=0.5, sigma_cut=8.0,
):
    """Return one ridge position on a sky arc, or None without qualifying emission."""
    settings = [radius_mas, noise_jy_per_beam, pa_center_deg,
                half_width_deg, angle_step_deg, sigma_cut]
    if not np.isfinite(settings).all():
        raise ValueError("Arc settings and noise must be finite.")
    if radius_mas <= 0 or noise_jy_per_beam <= 0 or angle_step_deg <= 0 or sigma_cut <= 0:
        raise ValueError("Radius, noise, angular step and sigma cut must be positive.")
    if not 0 < half_width_deg < 180:
        raise ValueError("Arc half-width must be between 0 and 180 degrees.")
    if image.ndim != 2:
        raise ValueError("Arc sampling requires a 2D image.")
    if str(header.get("BUNIT", "")).replace(" ", "").upper() != "JY/BEAM":
        raise ValueError("Arc sampling requires image units of Jy/beam.")

    sample_count = int(np.ceil(2 * half_width_deg / angle_step_deg)) + 1
    angles = np.linspace(pa_center_deg - half_width_deg, pa_center_deg + half_width_deg, sample_count)
    wcs = WCS(header).celestial
    height, width = image.shape
    core = wcs.pixel_to_world((width - 1) / 2, (height - 1) / 2)
    positions = core.directional_offset_by(angles * u.deg, radius_mas * u.mas)
    x_pixel, y_pixel = wcs.world_to_pixel(positions)
    inside = (
        np.isfinite(x_pixel) & np.isfinite(y_pixel)
        & (x_pixel >= 0) & (x_pixel <= width - 1)
        & (y_pixel >= 0) & (y_pixel <= height - 1)
    )
    brightness = map_coordinates(
        image, [y_pixel, x_pixel], order=1, output=np.float64,
        mode="constant", cval=np.nan, prefilter=False,
    )
    detected = inside & np.isfinite(brightness) & (brightness >= sigma_cut * noise_jy_per_beam)
    if not detected.any():
        return None

    # Equal angular spacing at fixed radius gives equal arc-length weights.
    weights = np.where(detected, brightness, 0.0)
    index = int(np.searchsorted(np.cumsum(weights), weights.sum() / 2))
    return {
        "radius_mas": float(radius_mas),
        "pa_deg": float(angles[index] % 360),
        "x_pixel": float(x_pixel[index]),
        "y_pixel": float(y_pixel[index]),
        "arc_clipped": bool(not inside.all()),
    }


In [ ]:
# Increasing-radius arc tracing follows Pushkarev et al. (2017), Section 3.1.
# https://arxiv.org/pdf/1705.02888
# Implementation choices: 0.05 mas raw radial steps, a moving angular
# window centred on the previous point, and stopping at the first undetected arc.
# The paper's approximately 0.05 mas spacing refers to its interpolated spline.

def trace_jet_ridgeline(
    image, header, noise_jy_per_beam, initial_pa_deg, radial_step_mas=0.05,
    max_radius_mas=None, half_width_deg=90.0, angle_step_deg=0.5, sigma_cut=8.0,
):
    """Return sampled ridge points and a stopping reason; images are not retained."""
    if image.ndim != 2:
        raise ValueError("Ridgeline tracing requires a 2D image.")
    if not np.isfinite(initial_pa_deg):
        raise ValueError("A finite initial jet position angle is required.")
    if not np.isfinite(radial_step_mas) or radial_step_mas <= 0:
        raise ValueError("Radial step must be finite and positive, in mas.")

    stopping_reason = "max_radius"
    if max_radius_mas is None:
        height, width = image.shape
        wcs = WCS(header).celestial
        core = wcs.pixel_to_world((width - 1) / 2, (height - 1) / 2)
        corners = wcs.pixel_to_world(
            [0, width - 1, 0, width - 1], [0, 0, height - 1, height - 1],
        )
        max_radius_mas = float(core.separation(corners).mas.max())
        stopping_reason = "image_radius_limit"
    if not np.isfinite(max_radius_mas) or max_radius_mas < radial_step_mas:
        raise ValueError("Maximum radius must be finite and at least one radial step.")

    points = []
    pa_center = float(initial_pa_deg)
    step_count = int(np.floor(max_radius_mas / radial_step_mas))
    for step in range(1, step_count + 1):
        point = sample_jet_arc(
            image, header, step * radial_step_mas, noise_jy_per_beam, pa_center,
            half_width_deg=half_width_deg, angle_step_deg=angle_step_deg, sigma_cut=sigma_cut,
        )
        if point is None:
            stopping_reason = "no_detection"
            break
        points.append(point)
        pa_center = point["pa_deg"]

    ridge = pd.DataFrame(points, columns=[
        "radius_mas", "pa_deg", "x_pixel", "y_pixel", "arc_clipped",
    ])
    return ridge, stopping_reason


In [ ]:
from scipy.interpolate import CubicSpline

# Pushkarev et al. (2017), Section 3.1: cubic-spline ridge interpolation
# at approximately 0.05 mas radial intervals.
# https://arxiv.org/pdf/1705.02888
# Numerical choices: interpolate pixel x and y against sampled arc radius,
# with natural boundary conditions and no extrapolation.
# The radius parameter labels the interpolation grid; intermediate spline
# positions are not new detections or independent radial measurements.

def interpolate_ridgeline(ridge, radial_step_mas=0.05):
    """Interpolate within the detected ridge range, preserving both endpoints."""
    if not np.isfinite(radial_step_mas) or radial_step_mas <= 0:
        raise ValueError("Interpolation step must be finite and positive, in mas.")
    points = ridge[["radius_mas", "x_pixel", "y_pixel"]].copy()
    points = points.rename(columns={"radius_mas": "radius_parameter_mas"})
    values = points.to_numpy(dtype=np.float64)
    if not np.isfinite(values).all():
        raise ValueError("Ridgeline coordinates and radii must be finite.")
    radius = values[:, 0]
    if (np.diff(radius) <= 0).any():
        raise ValueError("Ridgeline radii must be strictly increasing.")
    if len(points) < 2:
        return points.reset_index(drop=True)

    step_count = int(np.floor((radius[-1] - radius[0]) / radial_step_mas))
    sample_radius = radius[0] + radial_step_mas * np.arange(step_count + 1)
    sample_radius = np.append(sample_radius[sample_radius < radius[-1]], radius[-1])
    spline = CubicSpline(
        radius, values[:, 1:], axis=0, bc_type="natural", extrapolate=False,
    )
    coordinates = spline(sample_radius)
    return pd.DataFrame({
        "radius_parameter_mas": sample_radius,
        "x_pixel": coordinates[:, 0],
        "y_pixel": coordinates[:, 1],
    })


In [ ]:
# Modification of Pushkarev et al. (2017), Section 3.1:
# report radial core-to-endpoint separation, rather than ridgeline path length.
# https://arxiv.org/pdf/1705.02888
# The endpoint is the last raw arc detection; spline excursions do not extend it.
# This is a beam- and threshold-dependent detected extent, not a deconvolved size.
# The core is fixed at the geometric image centre.

def jet_radial_extent_mas(ridge, header):
    """Return centre-to-final-detection separation in mas, or NaN for an empty trace."""
    if ridge.empty:
        return float("nan")

    endpoint = ridge.iloc[-1]
    x_pixel = float(endpoint["x_pixel"])
    y_pixel = float(endpoint["y_pixel"])
    width, height = header["NAXIS1"], header["NAXIS2"]
    if not np.isfinite([x_pixel, y_pixel]).all():
        raise ValueError("Ridgeline endpoint coordinates must be finite.")
    if not (0 <= x_pixel <= width - 1 and 0 <= y_pixel <= height - 1):
        raise ValueError("Ridgeline endpoint lies outside the image.")

    wcs = WCS(header).celestial
    core = wcs.pixel_to_world((width - 1) / 2, (height - 1) / 2)
    final_position = wcs.pixel_to_world(x_pixel, y_pixel)
    extent = float(core.separation(final_position).mas)
    if not np.isfinite(extent):
        raise ValueError("Could not determine a finite sky separation for the endpoint.")
    return extent


In [ ]:
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait
from itertools import islice

# Each worker loads one image and returns scalar measurements and metadata.
# At most 'workers' image jobs are pending; results retain selection order.
if not isinstance(workers, int) or workers < 1:
    raise ValueError("workers must be a positive integer.")
if any(mode not in ("native", "circular") for mode in extent_beam_modes):
    raise ValueError("Extent beam modes must be 'native' or 'circular'.")

def measure_preview_image(record):
    row = record.copy()
    filename = row["image_filename"]
    row["error"] = ""
    row.update({f"orientation_{key}": value for key, value in orientation_settings.items()})
    row.update({f"ridgeline_{key}": value for key, value in ridgeline_settings.items()})
    image = header = mode_image = mode_header = None
    try:
        image, header = load_image(filename)
        row.update(beam_noise_metadata(header))
        row["total_flux_jy"] = total_flux_jy(filename)
        row["peak_brightness_jy_per_beam"] = peak_brightness_jy_per_beam(image, header)
        row["jet_pa_deg"] = (
            jet_orientation_deg(filename, row["noise_jy_per_beam"], **orientation_settings)
            if np.isfinite(row["noise_jy_per_beam"]) else np.nan
        )
        row["extent_initial_pa_deg"] = (
            row["jet_pa_deg"] if initial_pa_override_deg is None else float(initial_pa_override_deg)
        )

        for mode in extent_beam_modes:
            prefix = f"extent_{mode}"
            row[f"jet_radial_extent_{mode}_mas"] = np.nan
            row[f"{prefix}_stop_reason"] = "unavailable_direction_or_noise"
            mode_image, mode_header = (
                (image, header) if mode == "native" else circularise_image(image, header)
            )
            rms = four_corner_rms_jy_per_beam(mode_image, mode_header)
            row[f"{prefix}_noise_jy_per_beam"] = rms
            row[f"{prefix}_noise_method"] = "min_four_corner_rms_about_zero"
            row[f"{prefix}_bmaj_mas"] = mode_header["BMAJ"] * 3_600_000
            row[f"{prefix}_bmin_mas"] = mode_header["BMIN"] * 3_600_000
            row[f"{prefix}_bpa_deg"] = mode_header["BPA"]
            if not np.isfinite(row["extent_initial_pa_deg"]) or not np.isfinite(rms):
                continue

            ridge, reason = trace_jet_ridgeline(
                mode_image, mode_header, rms, row["extent_initial_pa_deg"], **ridgeline_settings,
            )
            row[f"jet_radial_extent_{mode}_mas"] = jet_radial_extent_mas(ridge, mode_header)
            row[f"{prefix}_stop_reason"] = reason
            row[f"{prefix}_arc_clipped"] = bool(ridge["arc_clipped"].any())
            if not ridge.empty:
                row[f"{prefix}_endpoint_x_pixel"] = float(ridge.iloc[-1]["x_pixel"])
                row[f"{prefix}_endpoint_y_pixel"] = float(ridge.iloc[-1]["y_pixel"])
    except Exception as error:
        row["error"] = f"{type(error).__name__}: {error}"
    finally:
        image = header = mode_image = mode_header = None
    return row

preview_rows = [None] * len(selected_files)
preview_jobs = iter(enumerate(
    record._asdict() for record in selected_files.itertuples(index=False)
))
with ThreadPoolExecutor(max_workers=workers) as executor:
    pending = {}
    for index, record in islice(preview_jobs, workers):
        pending[executor.submit(measure_preview_image, record)] = index
    while pending:
        completed, _ = wait(pending, return_when=FIRST_COMPLETED)
        for future in completed:
            index = pending.pop(future)
            preview_rows[index] = future.result()
            next_job = next(preview_jobs, None)
            if next_job is not None:
                next_index, next_record = next_job
                pending[executor.submit(measure_preview_image, next_record)] = next_index

preview_metrics = pd.DataFrame(preview_rows)
preview_columns = [
    "image_filename", "total_flux_jy", "peak_brightness_jy_per_beam", "jet_pa_deg",
    "extent_initial_pa_deg",
]
for mode in extent_beam_modes:
    preview_columns.extend([
        f"jet_radial_extent_{mode}_mas", f"extent_{mode}_noise_jy_per_beam",
        f"extent_{mode}_stop_reason", f"extent_{mode}_arc_clipped",
    ])
preview_columns.append("error")
with pd.option_context("display.max_colwidth", None):
    display(preview_metrics.reindex(columns=preview_columns))


In [ ]:
import warnings
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from matplotlib.transforms import Affine2D
from astropy.visualization import AsinhStretch, ImageNormalize
from astropy.wcs import FITSFixedWarning

# Local east/north offsets use the FITS WCS linear scale matrix,
# a tangent-plane approximation appropriate to these milliarcsecond fields.
# Scalar labels use preview_metrics. Ridge positions are recomputed for each
# displayed image and beam mode with the stored initial PA and noise.
def show_image_preview(filename, view_radius_mas=None):
    """Load one image on demand and compare its configured beam modes."""
    modes = list(extent_beam_modes)
    if not modes or any(mode not in ("native", "circular") for mode in modes):
        raise ValueError("Choose at least one valid extent beam mode.")
    matches = preview_metrics.loc[preview_metrics["image_filename"] == filename]
    metrics = matches.iloc[0].to_dict() if not matches.empty else {}
    print(filename)
    fields = ["total_flux_jy", "peak_brightness_jy_per_beam", "jet_pa_deg", "extent_initial_pa_deg"]
    fields += [f"jet_radial_extent_{mode}_mas" for mode in modes]
    display(pd.DataFrame([metrics]).reindex(columns=fields).fillna("unavailable"))
    if metrics.get("error", ""):
        print(f"Calculation error: {metrics['error']}")

    image, header = load_image(filename)
    beam = beam_noise_metadata(header)
    core_pixel = np.array([(image.shape[1] - 1) / 2, (image.shape[0] - 1) / 2])
    scale = WCS(header).celestial.pixel_scale_matrix * 3_600_000
    matrix = np.eye(3)
    matrix[:2, :2] = scale
    matrix[:2, 2] = -scale @ core_pixel
    pixel_to_mas = Affine2D(matrix)
    if view_radius_mas is None:
        extents = [metrics.get(f"jet_radial_extent_{mode}_mas", np.nan) for mode in modes]
        detected_max = max((value for value in extents if np.isfinite(value)), default=0.0)
        view_radius_mas = max(5.0, 3 * beam["bmaj_mas"], 1.25 * detected_max)
    if not np.isfinite(view_radius_mas) or view_radius_mas <= 0:
        raise ValueError("View radius must be finite and positive, in mas.")

    fig, axes = plt.subplots(1, len(modes), figsize=(6 * len(modes), 5),
                             squeeze=False, constrained_layout=True)
    fig.suptitle(filename)
    for ax, mode in zip(axes[0], modes):
        try:
            mode_image, mode_header = (
                (image, header) if mode == "native" else circularise_image(image, header)
            )
        except ValueError as error:
            ax.text(0.5, 0.5, f"{mode} image unavailable: {error}",
                    transform=ax.transAxes, ha="center", va="center", wrap=True)
            ax.set_axis_off()
            continue
        peak = max(float(np.nanmax(mode_image)), np.finfo(float).tiny)
        norm = ImageNormalize(vmin=0, vmax=peak, stretch=AsinhStretch(0.01), clip=True)
        artist = ax.imshow(mode_image, origin="lower", cmap="inferno", norm=norm,
                           interpolation="nearest", transform=pixel_to_mas + ax.transData)
        rms = metrics.get(f"extent_{mode}_noise_jy_per_beam", np.nan)
        detection_level = ridgeline_settings["sigma_cut"] * rms
        if np.isfinite(detection_level) and 0 < detection_level < peak:
            ax.contour(mode_image, levels=[detection_level], colors="white", linewidths=0.6,
                       linestyles=":", transform=pixel_to_mas + ax.transData)
        ax.plot(0, 0, "+", color="white", markersize=10, label="Core (image centre)")
        pa = metrics.get("jet_pa_deg", np.nan)
        if np.isfinite(pa):
            direction = np.deg2rad(pa)
            ax.annotate("", xy=(np.sin(direction), np.cos(direction)), xytext=(0, 0),
                        arrowprops={"arrowstyle": "->", "color": "cyan", "lw": 1.8})
            ax.plot([], [], color="cyan", label="Inner jet PA (1 mas arrow)")
        else:
            ax.text(0.02, 0.97, "PA unavailable", transform=ax.transAxes,
                    color="white", va="top")
        initial_pa = pa if initial_pa_override_deg is None else float(initial_pa_override_deg)
        if np.isfinite(initial_pa) and np.isfinite(rms) and rms > 0:
            with warnings.catch_warnings():
                warnings.filterwarnings(
                    "ignore", category=FITSFixedWarning,
                    message=r"'datfix' made the change 'Set MJD-OBS to .* from DATE-OBS'",
                )
                ridge, _ = trace_jet_ridgeline(
                    mode_image, mode_header, rms, initial_pa, **ridgeline_settings,
                )
            if not ridge.empty:
                raw_offsets = (ridge[["x_pixel", "y_pixel"]].to_numpy() - core_pixel) @ scale.T
                ax.plot(raw_offsets[:, 0], raw_offsets[:, 1], ".",
                        color="magenta", markersize=3, label="Detected ridge points")
                if len(ridge) >= 2:
                    curve = interpolate_ridgeline(ridge, radial_step_mas=ridge_interpolation_step_mas)
                    curve_offsets = (curve[["x_pixel", "y_pixel"]].to_numpy() - core_pixel) @ scale.T
                    ax.plot(curve_offsets[:, 0], curve_offsets[:, 1],
                            color="deepskyblue", lw=1.2, label="Interpolated ridgeline")
            else:
                ax.text(0.02, 0.83, "No ridgeline detected", transform=ax.transAxes,
                        color="white", va="top")
        else:
            ax.text(0.02, 0.83, "Ridgeline unavailable: direction or noise missing",
                    transform=ax.transAxes, color="white", va="top")
        endpoint = np.array([metrics.get(f"extent_{mode}_endpoint_x_pixel", np.nan),
                             metrics.get(f"extent_{mode}_endpoint_y_pixel", np.nan)])
        extent = metrics.get(f"jet_radial_extent_{mode}_mas", np.nan)
        if np.isfinite(endpoint).all() and np.isfinite(extent):
            east, north = scale @ (endpoint - core_pixel)
            ax.plot([0, east], [0, north], "o--", color="lime", markersize=4,
                    label=f"Radial extent: {extent:.2f} mas")
        else:
            ax.text(0.02, 0.90, "Extent unavailable", transform=ax.transAxes,
                    color="white", va="top")
        bmaj = mode_header.get("BMAJ", np.nan) * 3_600_000
        bmin = mode_header.get("BMIN", np.nan) * 3_600_000
        bpa = mode_header.get("BPA", np.nan)
        if np.isfinite([bmaj, bmin, bpa]).all() and min(bmaj, bmin) > 0:
            ax.add_patch(Ellipse((0.77 * view_radius_mas, -0.77 * view_radius_mas),
                                 width=bmaj, height=bmin, angle=90 - bpa,
                                 facecolor="white", edgecolor="white", alpha=0.8))
        else:
            ax.text(0.02, 0.03, "Beam unavailable", transform=ax.transAxes, color="white")
        clipped = metrics.get(f"extent_{mode}_arc_clipped", False)
        if pd.notna(clipped) and bool(clipped):
            ax.text(0.98, 0.03, "Arc window clipped", transform=ax.transAxes,
                    ha="right", color="white")
        ax.set(xlim=(view_radius_mas, -view_radius_mas),
               ylim=(-view_radius_mas, view_radius_mas), aspect="equal",
               xlabel="East offset (mas)", ylabel="North offset (mas)", title=f"{mode} beam")
        ax.set_facecolor("black")
        ax.legend(loc="upper right", fontsize=7, facecolor="black", labelcolor="white")
        fig.colorbar(artist, ax=ax, label="Jy/beam", fraction=0.046, pad=0.04)
    plt.show()
    plt.close(fig)

# Explicit filenames can be assigned here for repeated inspection.
inspection_filenames = preview_metrics["image_filename"].head(5).tolist()
for filename in inspection_filenames:
    show_image_preview(filename)


In [ ]:
inspection_filenames = preview_metrics["image_filename"].iloc[5:12].tolist()
for filename in inspection_filenames:
    show_image_preview(filename)